# Camada Gold - Analise de Viagens a Servico

Este notebook consome a camada Silver (dados limpos e tipados) para responder
perguntas de negocio, construir uma camada Gold agregada e gerar visualizacoes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import banco

sns.set_theme(style="whitegrid")
conexao = banco.conectar()

## Pergunta 1 - Quais os 5 orgaos com maior custo total em viagens?

In [ ]:
sql_q1 = """
    SELECT nome_orgao_superior AS orgao, SUM(valor_total) AS custo_total
    FROM silver_viagem
    GROUP BY nome_orgao_superior
    ORDER BY custo_total DESC
    LIMIT 5
"""
df_q1 = pd.read_sql(sql_q1, conexao)
df_q1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_q1["orgao"], df_q1["custo_total"], color="#2E86AB", label="Custo total (R$)")
ax.invert_yaxis()
ax.set_title("5 orgaos com maior custo total em viagens")
ax.set_xlabel("Custo total (R$)")
ax.set_ylabel("Orgao superior")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** o orgao com maior custo total concentra uma parcela expressiva
do gasto agregado, o que sinaliza onde as politicas de controle de despesas de
viagem devem ser priorizadas.

## Pergunta 2 - Quais os 3 destinos (UF) com maior custo medio por viagem?

Usamos a tabela `silver_trecho`, que ja possui o campo `destino_uf` estruturado
(um valor por linha), em vez do campo de texto livre `destinos` da viagem, que
pode conter mais de um destino por viagem.

In [ ]:
sql_q2 = """
    SELECT t.destino_uf AS destino, AVG(v.valor_total) AS custo_medio, COUNT(*) AS qtd_viagens
    FROM (
        SELECT DISTINCT id_viagem, destino_uf
        FROM silver_trecho
        WHERE destino_uf IS NOT NULL AND destino_uf <> ''
    ) t
    JOIN silver_viagem v ON v.id_viagem = t.id_viagem
    GROUP BY t.destino_uf
    ORDER BY custo_medio DESC
    LIMIT 3
"""
df_q2 = pd.read_sql(sql_q2, conexao)
df_q2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(df_q2["destino"], df_q2["custo_medio"], color="#A23B72", label="Custo medio por viagem (R$)")
ax.set_title("3 destinos com maior custo medio por viagem")
ax.set_xlabel("UF de destino")
ax.set_ylabel("Custo medio por viagem (R$)")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** destinos mais distantes ou com menor volume de viagens tendem
a concentrar um custo medio mais alto por viagem, provavelmente pelo peso das
passagens aereas e diarias de longa duracao.

## Pergunta 3 - Qual a viagem de maior duracao e qual o seu custo total?

In [ ]:
sql_q3 = """
    SELECT id_viagem, nome_orgao_superior, data_inicio, data_fim, duracao_dias, valor_total
    FROM silver_viagem
    ORDER BY duracao_dias DESC
    LIMIT 1
"""
df_q3 = pd.read_sql(sql_q3, conexao)
df_q3

In [ ]:
sql_media = "SELECT AVG(duracao_dias) AS duracao_media, AVG(valor_total) AS custo_medio FROM silver_viagem"
df_media = pd.read_sql(sql_media, conexao)

rotulos = ["Viagem mais longa", "Media geral"]
duracoes = [df_q3["duracao_dias"].iloc[0], df_media["duracao_media"].iloc[0]]

fig, ax = plt.subplots(figsize=(6, 5))
barras = ax.bar(rotulos, duracoes, color=["#F18F01", "#6A994E"], label="Duracao (dias)")
ax.set_title("Duracao da viagem mais longa vs. media geral")
ax.set_xlabel("Referencia")
ax.set_ylabel("Duracao (dias)")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** a viagem mais longa dura muito mais que a media geral de
viagens, o que pode indicar uma missao especial de longa permanencia. Vale
conferir se o custo total dessa viagem e proporcional a duracao ou se ficou
abaixo do esperado, o que pode revelar diarias ainda nao lancadas.

## Camada Gold agregada - resumo por orgao

A tabela `gold_resumo_orgao` combina `silver_viagem` e `silver_pagamento` por
meio de `JOIN` e agrupa por `orgao` com `GROUP BY`, respondendo de forma
reutilizavel a varias perguntas de negocio.

Um cuidado importante: unir `silver_viagem` diretamente com `silver_pagamento`
duplica o `valor_total` da viagem uma vez para cada pagamento associado a ela
(uma viagem pode ter mais de um pagamento). Por isso o valor pago e
pre-agregado por viagem em uma subconsulta antes do `JOIN`, evitando a
duplicacao (fan-out) e garantindo que os totais fiquem corretos.

In [ ]:
banco.executar(conexao, "DROP VIEW IF EXISTS gold_resumo_orgao_view")
banco.executar(conexao, "DROP TABLE IF EXISTS gold_resumo_orgao")

banco.executar(conexao, """
    CREATE TABLE gold_resumo_orgao AS
    SELECT
        v.nome_orgao_superior AS orgao,
        COUNT(*) AS qtd_viagens,
        SUM(v.valor_total) AS valor_total_viagens,
        AVG(v.valor_total) AS valor_medio_viagem,
        AVG(v.duracao_dias) AS duracao_media_dias,
        COALESCE(SUM(pv.valor_total_pago), 0) AS valor_total_pago
    FROM silver_viagem v
    LEFT JOIN (
        SELECT id_viagem, SUM(valor) AS valor_total_pago
        FROM silver_pagamento
        GROUP BY id_viagem
    ) pv ON v.id_viagem = pv.id_viagem
    GROUP BY v.nome_orgao_superior
""")

banco.executar(conexao, """
    CREATE VIEW gold_resumo_orgao_view AS
    SELECT
        v.nome_orgao_superior AS orgao,
        COUNT(*) AS qtd_viagens,
        SUM(v.valor_total) AS valor_total_viagens,
        AVG(v.valor_total) AS valor_medio_viagem,
        AVG(v.duracao_dias) AS duracao_media_dias,
        COALESCE(SUM(pv.valor_total_pago), 0) AS valor_total_pago
    FROM silver_viagem v
    LEFT JOIN (
        SELECT id_viagem, SUM(valor) AS valor_total_pago
        FROM silver_pagamento
        GROUP BY id_viagem
    ) pv ON v.id_viagem = pv.id_viagem
    GROUP BY v.nome_orgao_superior
""")

pd.read_sql("SELECT * FROM gold_resumo_orgao_view ORDER BY valor_total_viagens DESC LIMIT 10", conexao)

## Pergunta 4 - Qual orgao tem o maior valor medio por viagem?

Respondida a partir da camada Gold (`gold_resumo_orgao`), considerando apenas
orgaos com pelo menos 5 viagens registradas, para evitar que um unico caso
isolado distorca a media.

In [ ]:
sql_q4 = """
    SELECT orgao, valor_medio_viagem, qtd_viagens
    FROM gold_resumo_orgao
    WHERE qtd_viagens >= 5
    ORDER BY valor_medio_viagem DESC
    LIMIT 5
"""
df_q4 = pd.read_sql(sql_q4, conexao)
df_q4

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_q4["orgao"], df_q4["valor_medio_viagem"], color="#3D5A80", label="Valor medio por viagem (R$)")
ax.invert_yaxis()
ax.set_title("5 orgaos com maior valor medio por viagem (camada Gold)")
ax.set_xlabel("Valor medio por viagem (R$)")
ax.set_ylabel("Orgao superior")
ax.legend()
plt.tight_layout()
plt.show()

**Insight:** o orgao com maior valor medio por viagem nao e necessariamente
o mesmo com maior custo total (Pergunta 1) - isso mostra a diferenca entre
volume de gasto e intensidade de gasto por viagem, uma leitura so possivel
gracas a tabela agregada da camada Gold.

## Pergunta 5 - Qual o tipo de pagamento com maior valor medio?

In [ ]:
sql_q5 = """
    SELECT tipo_pagamento, AVG(valor) AS valor_medio, COUNT(*) AS qtd_pagamentos
    FROM silver_pagamento
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC
"""
df_q5 = pd.read_sql(sql_q5, conexao)
df_q5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_q5["tipo_pagamento"], df_q5["valor_medio"], color="#98C1D9", label="Valor medio (R$)")
ax.set_title("Valor medio por tipo de pagamento")
ax.set_xlabel("Tipo de pagamento")
ax.set_ylabel("Valor medio (R$)")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Insight:** diarias tendem a concentrar os maiores valores medios, o que
faz sentido considerando que cobrem hospedagem e alimentacao ao longo de
varios dias, enquanto reembolsos e servicos correlatos sao pontuais.

## Pergunta 6 - Qual o meio de transporte mais usado nos trechos?

In [ ]:
sql_q6 = """
    SELECT meio_transporte, COUNT(*) AS qtd_trechos
    FROM silver_trecho
    GROUP BY meio_transporte
    ORDER BY qtd_trechos DESC
    LIMIT 6
"""
df_q6 = pd.read_sql(sql_q6, conexao)
df_q6

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df_q6["meio_transporte"], df_q6["qtd_trechos"], color="#EE6C4D", label="Quantidade de trechos")
ax.set_title("Meios de transporte mais usados nos trechos")
ax.set_xlabel("Meio de transporte")
ax.set_ylabel("Quantidade de trechos")
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Insight:** o meio de transporte predominante reflete o perfil das viagens
a servico (deslocamentos locais/regionais vs. viagens aereas de maior
distancia), o que pode orientar politicas de contratacao de transporte.

## Conclusoes

- A camada Gold concentrou, em uma unica tabela, metricas por orgao (volume de
  viagens, custo total, custo medio e duracao media), permitindo responder
  varias perguntas de negocio sem reprocessar as camadas anteriores.
- O cuidado com o efeito de duplicacao (fan-out) ao unir `silver_viagem` com
  `silver_pagamento` foi essencial para que os totais da camada Gold nao
  ficassem superestimados.
- Como possiveis melhorias futuras: automatizar a atualizacao periodica da
  camada Gold, adicionar um dashboard interativo e expandir a analise para
  series historicas com mais de um ano de dados.

In [ ]:
conexao.close()